Import Libraries

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

Load Data & Filter Features

In [17]:
# 1. Load the dataset
df = pd.read_csv("mushrooms.csv")
df

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8119,e,k,s,n,f,n,a,c,b,y,...,s,o,o,p,o,o,p,b,c,l
8120,e,x,s,n,f,n,a,c,b,y,...,s,o,o,p,n,o,p,b,v,l
8121,e,f,s,n,f,n,a,c,b,n,...,s,o,o,p,o,o,p,b,c,l
8122,p,k,y,n,f,y,f,c,n,b,...,k,w,w,p,w,o,e,w,v,l


In [18]:


print("--- INITIAL PREVIEW ---")
display(df.head())

# 2. CHECK FOR MISSING VALUES (Null/NaN)
print("\n--- MISSING VALUES CHECK ---")
missing_counts = df.isnull().sum()
total_missing = missing_counts.sum()
print("Missing:",missing_counts)




--- INITIAL PREVIEW ---


,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g



--- MISSING VALUES CHECK ---
Missing: class                       0
cap-shape                   0
cap-surface                 0
cap-color                   0
bruises                     0
odor                        0
gill-attachment             0
gill-spacing                0
gill-size                   0
gill-color                  0
stalk-shape                 0
stalk-root                  0
stalk-surface-above-ring    0
stalk-surface-below-ring    0
stalk-color-above-ring      0
stalk-color-below-ring      0
veil-type                   0
veil-color                  0
ring-number                 0
ring-type                   0
spore-print-color           0
population                  0
habitat                     0
dtype: int64


In [19]:
# 3. CHECK FOR DUPLICATE ROWS
print("\n--- DUPLICATE ROWS CHECK ---")
duplicate_count = df.duplicated().sum()
print("Duplicates:", duplicate_count)



# 4. TARGET DISTRIBUTION CHECK
print("\n--- TARGET CLASS DISTRIBUTION ---")
print(df['class'].value_counts())


--- DUPLICATE ROWS CHECK ---
Duplicates: 0

--- TARGET CLASS DISTRIBUTION ---
class
e    4208
p    3916
Name: count, dtype: int64


In [26]:
#-'odor': ក្លិន (odor: almond=a,anise=l,creosote=c,fishy=y,foul=f,musty=m,none=n,pungent=p,spicy=s)
#-'gill-size': ទំហំស្រទាប់ក្រោមមួកផ្សិត (gill-size: broad=b,narrow=n)
#-'gill-color': ពណ៌ស្រទាប់ក្រោមមួកផ្សិត (gill-color: black=k,brown=n,gray=g,green=p,orange=o,purple=u,red=r,white=w,yellow=y)
#-'stalk-surface-above-ring': សណ្ឋានផ្ទៃដើមផ្សិតផ្នែកខាងលើកង (stalk-surface-above-ring: fibrous=f,scaly=y,silky=k,smooth=s)
#-'spore-print-color': ពណ៌លម្អង (spore-print-color: black=k,brown=n,gray=g,green=p,orange=o,purple=u,red=r,white=w,yellow=y)

In [ ]:
top_features = ['odor', 'gill-size', 'gill-color', 'stalk-surface-above-ring', 'spore-print-color']
X_raw = df[top_features]
y_raw = df['class']

Data Preprocessing & Categorical Encoding

In [21]:
encoders = {}
X = pd.DataFrame()

# Encode features and store mappings for the API
for col in top_features:
    le = LabelEncoder()
    X[col] = le.fit_transform(X_raw[col])
    encoders[col] = le 

# Encode target label (e -> 0, p -> 1)
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y_raw)

Train-Test Split

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

Hyperparameter Tuning with GridSearchCV

In [23]:
rf_base = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators':[50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
print("Tuning hyperparameters across combinations...")
grid_search.fit(X_train, y_train)

# Extract optimized model configuration
best_model = grid_search.best_estimator_
print(f"Optimal Parameters: {grid_search.best_params_}")

Tuning hyperparameters across combinations...
Optimal Parameters: {'criterion': 'gini', 'max_depth': 10, 'n_estimators': 50}


 Model Evaluation

In [24]:
predictions = best_model.predict(X_test)
print(f"Tuned Model Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=target_encoder.classes_))


Tuned Model Accuracy: 99.82%

Classification Report:
              precision    recall  f1-score   support

           e       1.00      1.00      1.00       842
           p       1.00      1.00      1.00       783

    accuracy                           1.00      1625
   macro avg       1.00      1.00      1.00      1625
weighted avg       1.00      1.00      1.00      1625



Save Files for Deployment

In [25]:
joblib.dump(best_model, "mushroom_model.pkl")
joblib.dump(encoders, "mushroom_encoders.pkl")
print("Model and encoders successfully saved!")

Model and encoders successfully saved!
